Selection Pocking video training FERAL test


Import
--

In [ ]:
import os
import cv2
import pandas as pd
from pathlib import Path
import numpy as np

---
CONFIGURATION
---

In [ ]:

# ------------ CONFIGURE HERE ------------
CSV_FOLDER = r"D:\MasterThesis\Data\Boris_annotation\Hope"   # folder with CSV

VIDEO_FOLDER_CAM1 = r"D:\MasterThesis\Data\videos\Chimps\Leipzig\group_B\cam1"      # folder with MP4
VIDEO_FOLDER_CAM2 = r"D:\MasterThesis\Data\videos\Chimps\Leipzig\group_B\cam2"      # folder with MP4


OUTPUT_FOLDER_BASE = r"D:\MasterThesis\Data\Behavior_of_Interest\poking\poking_clips_Hope_writing"

# Two subfolders, one per camera
OUTPUT_FOLDER_CAM1 = os.path.join(OUTPUT_FOLDER_BASE, "cam1")
OUTPUT_FOLDER_CAM2 = os.path.join(OUTPUT_FOLDER_BASE, "cam2")


TRACKING_CSV = os.path.join(OUTPUT_FOLDER_BASE, "poking_tracking_Hope_wrinting.csv")

#CSV_FILENAME = "Daza_4.8.22_Test3_T1_AE.csv"           # one CSV to start

COL_VIDEO = "Observation id"                # column containing video filename
COL_MODIFIER = "Modifier #1"        # column with behavior labels
COL_IMG_START = "Image index start"  # starting frame index for poking

# Ensure they exist
os.makedirs(OUTPUT_FOLDER_BASE, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_CAM1, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_CAM2, exist_ok=True)

#global dict to store per-(csv, video) sync offsets
# key = (csv_filename, video_name)  →  value = (offset_cam1, offset_cam2)

sync_offsets = {}

# ---------------------------------------


Functions
-

In [ ]:
def find_dual_video_paths(video_base: str) -> tuple[str, str]:
    """
    Find matching videos in BOTH cam1 and cam2 folders.
    Returns (cam1_path, cam2_path) or raises error if any missing.
    """
    # Construct expected filename (e.g. "Daza_18.8.22_Test6_T1.MP4")
    video_name = video_base + ".MP4"
    lower_target = video_name.lower()  # "daza_18.8.22_test6_t1.mp4"
    
    # ================= CAM1 =================
    cam1_candidate = os.path.join(VIDEO_FOLDER_CAM1, video_name)
    if not os.path.isfile(cam1_candidate):
        # Case-insensitive fallback for cam1
        cam1_candidate = None
        for f in os.listdir(VIDEO_FOLDER_CAM1):
            if f.lower() == lower_target:
                cam1_candidate = os.path.join(VIDEO_FOLDER_CAM1, f)
                break
    
    # ================= CAM2 =================  
    cam2_candidate = os.path.join(VIDEO_FOLDER_CAM2, video_name)
    if not os.path.isfile(cam2_candidate):
        # Case-insensitive fallback for cam2
        cam2_candidate = None
        for f in os.listdir(VIDEO_FOLDER_CAM2):
            if f.lower() == lower_target:
                cam2_candidate = os.path.join(VIDEO_FOLDER_CAM2, f)
                break
    
    # ================= VALIDATE BOTH FOUND =================
    if cam1_candidate is None:
        raise FileNotFoundError(f"CAM1 video missing in {VIDEO_FOLDER_CAM1}: {video_name}")
    if cam2_candidate is None:
        raise FileNotFoundError(f"CAM2 video missing in {VIDEO_FOLDER_CAM2}: {video_name}")
    
    print(f"  ✅ DUAL VIDEO: cam1/{os.path.basename(cam1_candidate)} + cam2/{os.path.basename(cam2_candidate)}")
    return cam1_candidate, cam2_candidate  # ← RETURNS TUPLE (cam1_path, cam2_path)

In [ ]:
def get_dual_video_props(cam1_path: str, cam2_path: str) -> tuple:
    """
    NEW: Load properties for BOTH cameras, validate FPS match.
    Returns (cap1, cap2, fps, total_frames) - assumes synchronized videos.
    """
    cap1 = cv2.VideoCapture(cam1_path)
    cap2 = cv2.VideoCapture(cam2_path)
    
    if not cap1.isOpened() or not cap2.isOpened():
        raise RuntimeError(f"Cannot open dual video: cam1={cam1_path}, cam2={cam2_path}")
    
    fps1 = cap1.get(cv2.CAP_PROP_FPS)
    fps2 = cap2.get(cv2.CAP_PROP_FPS)
    total1 = int(cap1.get(cv2.CAP_PROP_FRAME_COUNT))
    total2 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # ⚠️  CRITICAL: Validate synchronization
    if abs(fps1 - fps2) > 0.1:  # Allow 0.1 fps tolerance
        print(f"⚠️  FPS mismatch: cam1={fps1:.1f}, cam2={fps2:.1f}")
    if abs(total1 - total2) > 10:  # Allow small frame count diff
        print(f"⚠️  Frame count mismatch: cam1={total1}, cam2={total2}")
    
    fps = (fps1 + fps2) / 2  # Average FPS for playback
    total_frames = min(total1, total2)  # Use shorter duration
    
    return (cap1, cap2), fps, total_frames

In [ ]:
def play_dual_segment(caps: tuple,
                      start_frame: int,
                      end_frame: int,
                      fps: float,
                      offset_cam1: int = 0,
                      offset_cam2: int = 0):
    """
    Synchronized playback of cam1 and cam2 in TWO separate, persistent windows,
    plus a 3rd 'info' window. Windows are not destroyed on each replay.

    Uses per-camera offsets (offset_cam1, offset_cam2) to correct for
    small frame "déclage" between videos.

    Returns (key_char, new_offset_cam1, new_offset_cam2).
    """
    cap1, cap2 = caps
    print(f"\n🎬 DUAL PLAY: frames {start_frame}→{end_frame} ({end_frame-start_frame+1} frames)")
    print("V=Validate | M=Manual | B=Start+/- | E=End+/- | "
          "R=Restart | H=Hide | Y/U=Sync | Q=Quit")

    delay = int(1000 / fps) if fps > 0 else 33

    while True:
        # Apply current offsets when positioning streams
        cap1.set(cv2.CAP_PROP_POS_FRAMES, max(0, start_frame + offset_cam1))
        cap2.set(cv2.CAP_PROP_POS_FRAMES, max(0, start_frame + offset_cam2))

        segment_done = False
        while True:
            # Logical current frame = cam1 position minus its offset
            current1 = int(cap1.get(cv2.CAP_PROP_POS_FRAMES))
            logical_current = current1 - offset_cam1

            if logical_current > end_frame:
                segment_done = True
                break

            ret1, frame1 = cap1.read()
            ret2, frame2 = cap2.read()
            if not ret1 or not ret2:
                segment_done = True
                break

            # --- INFO WINDOW CONTENT ---
            txt = (f"[{start_frame}-{end_frame}] current {logical_current} "
                   f"| V/M/B/E/R/H/Q/U (U=sync)")
            info_h, info_w = 80, 800
            info_img = np.zeros((info_h, info_w, 3), dtype=np.uint8)
            cv2.putText(info_img, txt, (20, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            # Show frames + info in already existing windows
            cv2.imshow(WIN_CAM1, frame1)
            cv2.imshow(WIN_CAM2, frame2)
            cv2.imshow(WIN_INFO, info_img)

            key = cv2.waitKey(delay) & 0xFF
            if key in [ord('v'), ord('V'), ord('m'), ord('M'),
                       ord('b'), ord('B'), ord('e'), ord('E'),
                       ord('r'), ord('R'), ord('h'), ord('H'),
                       ord('q'), ord('Q'), ord('y'), ord('Y'),
                       ord('u'), ord('U')]:
                key_char = chr(key).lower()
                return key_char, offset_cam1, offset_cam2

        if segment_done:
            print("\n⏹️  End reached. Choose command (V/M/B/E/R/H/Q/Y/U).")
            while True:
                key = cv2.waitKey(0) & 0xFF
                if key in [ord('v'), ord('V'), ord('m'), ord('M'),
                           ord('b'), ord('B'), ord('e'), ord('E'),
                           ord('r'), ord('R'), ord('h'), ord('H'),
                           ord('q'), ord('Q'), ord('y'), ord('Y'),
                           ord('u'), ord('U')]:
                    key_char = chr(key).lower()
                    return key_char, offset_cam1, offset_cam2

In [ ]:
def cut_and_save_segment(input_path: str, start_frame: int, end_frame: int,
                         fps: float, output_path: str):
    """
    Cut [start_frame, end_frame] from input_path and save to output_path.
    """
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video for cutting: {input_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frame_count = 0
    while True:
        current = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        if current > end_frame:
            break

        ret, frame = cap.read()
        if not ret:
            break


        out.write(frame)
        frame_count += 1 

    cap.release()
    out.release()
    print(f"📹 Extracted {frame_count} clean frames to {output_path}")


def Main
-

In [ ]:
def main():
    # Create persistent windows once
    cv2.namedWindow(WIN_CAM1, cv2.WINDOW_NORMAL)
    cv2.namedWindow(WIN_CAM2, cv2.WINDOW_NORMAL)
    cv2.namedWindow(WIN_INFO, cv2.WINDOW_NORMAL)

    total_poking_count = 0
    csv_files = sorted([p for p in Path(CSV_FOLDER).iterdir()
                       if p.suffix.lower() == '.csv'])

    print(f"📁 Found {len(csv_files)} CSV files in {CSV_FOLDER}")
    for csv_file in csv_files:
        df_temp = pd.read_csv(str(csv_file))
        poking_in_csv = df_temp[df_temp[COL_MODIFIER] == "poking"]
        total_poking_count += len(poking_in_csv)
        print(f"   {csv_file.name}: {len(poking_in_csv)} poking rows")

    print(f"\n🎯 TOTAL: {total_poking_count} behavioral clips to validate")
    print(f"{'='*70}\n")

    if total_poking_count == 0:
        print("❌ No poking clips found anywhere!")
        return

    tracking_records = []
    processed_count = 0

    for csv_file in csv_files:
        csv_path = str(csv_file)
        csv_filename = os.path.basename(csv_path)
        print(f"\n{'='*60}")
        print(f"📄 Processing CSV: {csv_file.name}")
        print(f"{'='*60}")

        df = pd.read_csv(csv_path)
        df_poking = df[df[COL_MODIFIER] == "poking"]

        if df_poking.empty:
            print(f"   No poking rows → skipping")
            continue

        for idx, row in df_poking.iterrows():
            processed_count += 1

            # FIRST: load video + define variables
            video_name = row[COL_VIDEO]
            original_start = int(row[COL_IMG_START])

            try:
                cam1_path, cam2_path = find_dual_video_paths(video_name)
            except FileNotFoundError as e:
                ...
                continue

            caps, fps, total_frames = get_dual_video_props(cam1_path, cam2_path)

            start_frame = max(0, original_start - 25)
            end_frame = min(total_frames - 1, original_start + 35)

            # NEW: per-(csv, video) sync key and offsets
            key_sync = (csv_filename, video_name)
            offset_cam1, offset_cam2 = sync_offsets.get(key_sync, (0, 0))

            remaining = total_poking_count - processed_count + 1
            print(f"\n[{processed_count}/{total_poking_count}] 🎥 {video_name} | original frame {original_start}")
            print(f"   ⏳ Remaining to review: {remaining} clips")
            print(f"   | working frames {start_frame} → {end_frame}")
            print(f"{'='*60}")

            while True:
                print(f"🔄 [{video_name}] frames {start_frame} → {end_frame} (orig: {original_start})")
                print("V=Validate&Save | M=Manual | B=Start+/- | E=End+/- | "
                      "R=Replay | H=Hide/Skip | Y/U=Sync cams | Q=Quit")

                # NEW: pass current offsets and get possibly updated ones back
                key, offset_cam1, offset_cam2 = play_dual_segment(
                    caps, start_frame, end_frame, fps,
                    offset_cam1=offset_cam1,
                    offset_cam2=offset_cam2
                )
                # Store offsets for this (csv, video) so they persist for next rows
                sync_offsets[key_sync] = (offset_cam1, offset_cam2)

                print(f"→ User pressed: '{key}' "
                      f"(offsets: cam1={offset_cam1}, cam2={offset_cam2})")

                if key == 'u':
                    # ENTER SYNC MODE (from normal review or after reclipping)
                    offset_cam1, offset_cam2 = adjust_sync_offsets(offset_cam1, offset_cam2)
                    sync_offsets[key_sync] = (offset_cam1, offset_cam2)
                    # Re-run loop with new offsets
                    continue

                if key == 'q':
                    print(f"👋 QUIT at #{processed_count} | Remaining: {total_poking_count - processed_count}")
                    if 'caps' in locals():
                        for c in caps:
                            if c.isOpened():
                                c.release()
                    break
                
                elif key == 'v':
                    print(f"✅ VALIDATED #{processed_count} | Remaining: {total_poking_count - processed_count}")
                    base = os.path.splitext(os.path.basename(video_name))[0]
                    duration_frames = end_frame - start_frame + 1

                    out_name_cam1 = f"{base}_cam1_poking_{start_frame:06d}_{end_frame:06d}_{duration_frames:03d}f.mp4"
                    out_name_cam2 = f"{base}_cam2_poking_{start_frame:06d}_{end_frame:06d}_{duration_frames:03d}f.mp4"
                    
                    out_path_cam1 = os.path.join(OUTPUT_FOLDER_CAM1, out_name_cam1)
                    out_path_cam2 = os.path.join(OUTPUT_FOLDER_CAM2, out_name_cam2)
                    
                    # ===  ask which camera(s) are useful ===
                    print("Which cam is useful? [1] cam1 | [2] cam2 | [Enter] both")
                    cam_choice = input("Your choice (1/2 or Enter): ").strip()

                    if cam_choice == "1":
                        useful_cam = "cam1"
                    elif cam_choice == "2":
                        useful_cam = "cam2"
                    else:
                        useful_cam = "cam1,cam2"  

                    cam1_start_cut = max(0, start_frame + offset_cam1)
                    cam1_end_cut   = max(0, end_frame   + offset_cam1)

                    cam2_start_cut = max(0, start_frame + offset_cam2)
                    cam2_end_cut   = max(0, end_frame   + offset_cam2)

                    record = {
                        'video_name': base,
                        'csv_file': csv_filename,
                        'behavior': 'poking',
                        'original_img_start': original_start,
                        'final_img_start': start_frame,
                        'final_img_stop': end_frame,
                        'start_frame_diff': start_frame - original_start,
                        'end_frame_diff': end_frame - original_start, 
                        'behavior_frames': duration_frames,

                        'clip_filename_cam1': out_name_cam1,
                        'clip_filename_cam2': out_name_cam2,
                        'useful_cam': useful_cam,

                        'offset_cam1': offset_cam1,
                        'offset_cam2': offset_cam2,
                        'offset_diff_cam2_minus_cam1': offset_cam2 - offset_cam1,

                        'cam1_start_cut': cam1_start_cut,
                        'cam1_end_cut': cam1_end_cut,
                        'cam2_start_cut': cam2_start_cut,
                        'cam2_end_cut': cam2_end_cut,
                    }
                    tracking_records.append(record)
                    
                    print(f"💾 Saving CAM1 '{out_name_cam1}' and CAM2 '{out_name_cam2}' "
          f"({duration_frames} frames each)")
                    
                    cut_and_save_segment(cam1_path, cam1_start_cut, cam1_end_cut, fps, out_path_cam1)
                    cut_and_save_segment(cam2_path, cam2_start_cut, cam2_end_cut, fps, out_path_cam2)
                    
                    print(f"✅ Saved:\n   CAM1 → {out_path_cam1}\n   CAM2 → {out_path_cam2}")
                    
                    df_track = pd.DataFrame(tracking_records)
                    df_track.to_csv(TRACKING_CSV, index=False)
                    print(f"📊 Tracking updated: {len(tracking_records)} records")
                    
                    if 'caps' in locals():
                        for c in caps:
                            if c.isOpened():
                                c.release()

                    break

                elif key == 'h':
                    remarks = input("Remarks (or Enter for none): ").strip() or "bad_quality"
                    print(f"⏭️  HIDDEN #{processed_count} | Remaining: {total_poking_count - processed_count}")

                    base = video_name
                    record = {
                        'video_name': base,
                        'csv_file': csv_filename,
                        'behavior': 'poking_HIDDEN',
                        'original_img_start': original_start,
                        'final_img_start': -1,
                        'final_img_stop': -1,
                        'start_frame_diff': 0,
                        'end_frame_diff': 0,
                        'behavior_frames': 0,
                        'clip_filename_cam1': '',
                        'clip_filename_cam2': '',
                        'remarks': remarks
                    }
                    tracking_records.append(record)
                    
                    df_track = pd.DataFrame(tracking_records)
                    df_track.to_csv(TRACKING_CSV, index=False)
                    print(f"📊 Logged HIDDEN clip: '{remarks}' | Total: {len(tracking_records)} records")
                    
                    if 'caps' in locals():
                        for c in caps:
                             if c.isOpened():
                                 c.release()
                    break

                elif key == 'r':
                    print("🔄 Replaying current segment...")
                    continue

                elif key == 'm':
                    print("\n📝 MANUAL mode")
                    try:
                        new_start = int(input(f"New START frame (0–{total_frames-1}): "))
                        new_end = int(input(f"New END frame ({new_start}–{total_frames-1}): "))
                        new_start = max(0, min(new_start, total_frames - 1))
                        new_end = max(new_start, min(new_end, total_frames - 1))
                        start_frame, end_frame = new_start, new_end
                        print(f"✅ Updated: {start_frame} → {end_frame}")
                    except ValueError:
                        print("❌ Invalid numbers, keeping previous.")
                    continue

                elif key == 'b':
                    print("\n⬅️  Adjust START")
                    try:
                        offset = int(input("Offset (+/- frames): "))
                        new_start = max(0, min(start_frame + offset, end_frame))
                        start_frame = new_start
                        print(f"✅ New start: {start_frame}")
                    except ValueError:
                        print("❌ Invalid offset.")
                    continue

                elif key == 'e':
                    print("\n➡️  Adjust END")
                    try:
                        offset = int(input("Offset (+/- frames): "))
                        new_end = max(start_frame, min(end_frame + offset, total_frames - 1))
                        end_frame = new_end
                        print(f"✅ New end: {end_frame}")
                    except ValueError:
                        print("❌ Invalid offset.")
                    continue
            

    cv2.destroyAllWindows()

    # Optional final summary
    validated = sum(1 for r in tracking_records if r['behavior'] == 'poking')
    hidden = sum(1 for r in tracking_records if 'HIDDEN' in r['behavior'])
    print(f"\n🎉 FINISHED!")
    print(f"   ✅ Validated: {validated} clips")
    print(f"   ⏭️  Hidden: {hidden} clips")
    print(f"   📊 Total tracked: {len(tracking_records)} | {TRACKING_CSV}")


---
Synchronisation function
--


In [ ]:
def adjust_sync_offsets(offset_cam1: int, offset_cam2: int) -> tuple[int, int]:
    """
    Interactive synchronisation: choose cam (1/2) and +/- frames.
    Returns updated (offset_cam1, offset_cam2).

    This function is called when the user presses Y or U to align the two cameras.
    """
    print("\n🧭 SYNCHRONISATION MODE")
    print("Which camera do you want to shift?")
    print("1 = CAM1, 2 = CAM2, Enter = cancel")
    cam_choice = input("Cam to shift (1/2 or Enter): ").strip()

    if cam_choice not in ["1", "2"]:
        print("↩️  Sync cancelled.")
        return offset_cam1, offset_cam2

    try:
        delta = int(input("Offset (+/- frames, positive = later, negative = earlier): "))
    except ValueError:
        print("❌ Invalid number, sync cancelled.")
        return offset_cam1, offset_cam2

    if cam_choice == "1":
        offset_cam1 += delta
        print(f"✅ CAM1 offset is now {offset_cam1} frames.")
    else:
        offset_cam2 += delta
        print(f"✅ CAM2 offset is now {offset_cam2} frames.")

    return offset_cam1, offset_cam2

In [ ]:
# Global window names (constants)
WIN_CAM1 = "Poking validation - CAM1"
WIN_CAM2 = "Poking validation - CAM2"
WIN_INFO = "Poking validation - INFO"

RUN Here 
-

In [ ]:

if __name__ == "__main__":
    main()
    

📁 Found 24 CSV files in D:\MasterThesis\Data\Boris_annotation\Hope
   Hope_1.8.22_Test3_T2_AE.csv: 0 poking rows
   Hope_1.8.22_Test3_T3_AE.csv: 2 poking rows
   Hope_1.8.22_Test3_T4_AE.csv: 0 poking rows
   Hope_14.8.22_Test6_T1_AE.csv: 2 poking rows
   Hope_14.8.22_Test6_T2_AE.csv: 0 poking rows
   Hope_14.8.22_Test6_T3_AE.csv: 0 poking rows
   Hope_15.8.22_Test7_T1_AE.csv: 7 poking rows
   Hope_15.8.22_Test7_T2_AE.csv: 23 poking rows
   Hope_15.8.22_Test7_T3_AE.csv: 1 poking rows
   Hope_2.8.22_Test4_T1_AE.csv: 0 poking rows
   Hope_2.8.22_Test4_T2_AE.csv: 0 poking rows
   Hope_2.8.22_Test4_T3_AE.csv: 0 poking rows
   Hope_27.7.22_Hab_T1_AE.csv: 0 poking rows
   Hope_28.7.22_Hab_T2_AE.csv: 0 poking rows
   Hope_29.7.22_Test1_T1_AE.csv: 2 poking rows
   Hope_29.7.22_Test1_T2_AE.csv: 0 poking rows
   Hope_29.7.22_Test1_T3_AE.csv: 0 poking rows
   Hope_30.7.22_Test2_T1_AE.csv: 3 poking rows
   Hope_30.7.22_Test2_T2_AE.csv: 3 poking rows
   Hope_30.7.22_Test2_T3_AE.csv: 1 poking rows
  